Library

In [1]:
import pandas as pd 
import numpy as np
import medscheduler

print('libraries loaded')

libraries loaded


In [2]:
dir(medscheduler)

['AppointmentScheduler',
 'DEFAULT_AGE_GENDER_PROBS',
 'DEFAULT_FIRST_ATTENDANCE_RATIO',
 'DEFAULT_MONTH_WEIGHTS',
 'DEFAULT_STATUS_RATES',
 'DEFAULT_WEEKDAY_WEIGHTS',
 'Final',
 'PackageNotFoundError',
 'STATUS_KEYS',
 'Tuple',
 '__all__',
 '__annotations__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 'annotations',
 'constants',
 'pkg_version',
 'scheduler',
 'validate_defaults']

In [3]:
from medscheduler import AppointmentScheduler

In [4]:
help(AppointmentScheduler)

Help on class AppointmentScheduler in module medscheduler.scheduler:

class AppointmentScheduler(builtins.object)
 |  AppointmentScheduler(
 |      date_ranges: 'Optional[List[Tuple[datetime, datetime]]]' = None,
 |      ref_date: 'Optional[datetime]' = None,
 |      working_days: 'Optional[List[int]]' = None,
 |      appointments_per_hour: 'int' = 4,
 |      working_hours: 'Optional[List[Tuple[int, int]]]' = None,
 |      fill_rate: 'float' = 0.9,
 |      booking_horizon: 'int' = 30,
 |      median_lead_time: 'int' = 10,
 |      status_rates: 'Optional[Dict[str, float]]' = None,
 |      rebook_category: 'str' = 'med',
 |      check_in_time_mean: 'float' = -10.0,
 |      visits_per_year: 'float' = 1.2,
 |      first_attendance: 'float' = 0.325,
 |      bin_size: 'int' = 5,
 |      lower_cutoff: 'int' = 15,
 |      upper_cutoff: 'int' = 90,
 |      truncated: 'bool' = True,
 |      seed: 'Optional[int]' = 42,
 |      noise: 'float' = 0.1,
 |      age_gender_probs: 'Any' = ({'age_yrs': '

Generate Slots, Appointments, Patients

In [6]:
from medscheduler import AppointmentScheduler
from pathlib import Path
import pandas as pd
import numpy as np

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

scheduler = AppointmentScheduler(
    date_ranges=[("2024-01-01", "2024-12-31")],
    ref_date="2024-10-01",
    working_days=[0, 1, 2, 3, 4],
    working_hours=[(8, 17)],
    appointments_per_hour=4,
    fill_rate=0.80,
    booking_horizon=30,
    month_weights=[1] * 12,
    weekday_weights=[1] * 7,
    seed=42
)

slots_df, appointments_df, patients_df = scheduler.generate()

print("Slots:", slots_df.shape)
print("Appointments:", appointments_df.shape)
print("Patients:", patients_df.shape)

display(slots_df.head())
display(appointments_df.head())
display(patients_df.head())

Slots: (9432, 4)
Appointments: (6837, 16)
Patients: (5426, 4)


,slot_id,appointment_date,appointment_time,is_available
0,0001,2024-01-01,08:00:00,False
1,0002,2024-01-01,08:15:00,False
2,0003,2024-01-01,08:30:00,False
3,0004,2024-01-01,08:45:00,False
4,0005,2024-01-01,09:00:00,False


,appointment_id,slot_id,scheduling_date,appointment_date,appointment_time,scheduling_interval,status,check_in_time,appointment_duration,start_time,end_time,waiting_time,patient_id,sex,age,age_group
0,00006,0001,2023-12-08,2024-01-01,08:00:00,24,attended,07:35:33,46.5,08:01:05,08:47:35,25.5,00527,Female,23,20-24
1,00118,0021,2023-12-25,2024-01-01,13:00:00,7,did not attend,NaN,NaN,NaN,NaN,NaN,00320,Female,17,15-19
2,00194,0022,2023-12-29,2024-01-01,13:15:00,3,attended,13:01:40,16.9,13:28:46,13:45:40,27.1,00666,Female,67,65-69
3,00212,0023,2023-12-30,2024-01-01,13:30:00,2,attended,13:09:37,8.3,13:46:43,13:55:01,37.1,05154,Male,90,90+
4,00168,0024,2023-12-28,2024-01-01,13:45:00,4,attended,13:16:48,21.8,13:55:56,14:17:44,39.1,04135,Male,60,60-64


,patient_id,name,sex,dob
0,00001,Allison Hill,Female,1973-10-29
1,00002,Nancy Rhodes,Female,1944-01-23
2,00003,Angie Henderson,Female,1951-11-16
3,00004,Colleen Wagner,Female,1958-10-23
4,00005,Christina Santos,Female,1945-12-12


In [7]:
# Create synthetic doctors table
departments = {
    "Cardiology": 4,
    "Dermatology": 3,
    "General Medicine": 6,
    "Orthopedics": 4,
    "Pediatrics": 4,
    "Neurology": 3
}

doctor_rows = []
doctor_id = 1

for department, count in departments.items():
    for i in range(count):
        doctor_rows.append({
            "doctor_id": f"D{doctor_id:03d}",
            "doctor_name": f"Doctor {doctor_id}",
            "department": department,
            "max_appointments_per_day": np.random.choice([16, 20, 24, 28])
        })
        doctor_id += 1

doctors_df = pd.DataFrame(doctor_rows)

# Randomly assign department and doctor to each appointment
np.random.seed(42)

appointments_df["department"] = np.random.choice(
    list(departments.keys()),
    size=len(appointments_df),
    p=[0.15, 0.12, 0.30, 0.16, 0.17, 0.10]
)

def assign_doctor(department):
    available_doctors = doctors_df[doctors_df["department"] == department]["doctor_id"].tolist()
    return np.random.choice(available_doctors)

appointments_df["doctor_id"] = appointments_df["department"].apply(assign_doctor)

# Save CSV files
slots_df.to_csv(RAW_DIR / "slots.csv", index=False)
patients_df.to_csv(RAW_DIR / "patients.csv", index=False)
appointments_df.to_csv(RAW_DIR / "appointments.csv", index=False)
doctors_df.to_csv(RAW_DIR / "doctors.csv", index=False)

print("CSV files saved successfully in data/raw/")
print("Files created:")
print("- slots.csv")
print("- patients.csv")
print("- appointments.csv")
print("- doctors.csv")

CSV files saved successfully in data/raw/
Files created:
- slots.csv
- patients.csv
- appointments.csv
- doctors.csv
